## Setup

In [35]:
!python3 -m pip install pyarango
!python3 -m pip install "python-arango>=5.0" 


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [42]:
import json
import requests
import sys
import time                 
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

from arango.client import ArangoClient


# Initialize the client for ArangoDB.
client = ArangoClient(hosts="http://localhost:8529")


# Connect to "db_test" database as root user.
db = client.db("db_test", username="root", password="test")       

aql = db.aql

## First Graph

Recall

```
@prefix : <http://www.snee.com/ns/demo#> .
:Jane :hasParent :Gene .
:Gene :hasParent :Pat ;
      :gender    :female .
:Joan :hasParent :Pat ;
      :gender    :female .
:Pat  :gender    :male .
:Mike :hasParent :Joan .
```



In [43]:
# Create a new graph named "Family" if it does not already exist.
# This returns an API wrapper for "Family" graph.
if db.has_graph("Family"):
    familyGraph = db.graph("Family")
else:
    familyGraph = db.create_graph("Family")


# Create an edge definition named "Parent". This creates any missing
# collections and returns an API wrapper for "Parent" edge collection.
if not familyGraph.has_edge_definition("Parent"):
    parent = familyGraph.create_edge_definition(
        edge_collection="Parent",
        from_vertex_collections=["Person"],
        to_vertex_collections=["Person"]
    )
else:
    parent = familyGraph.edge_collection("Parent")

# List edge definitions.
familyGraph.edge_definitions()

[{'edge_collection': 'Parent',
  'from_vertex_collections': ['Person'],
  'to_vertex_collections': ['Person']}]

In [44]:
# Create a new vertex collection named "Person" if it does not exist.
# This returns an API wrapper for "Person" vertex collection.
if familyGraph.has_vertex_collection(name="Person"):
    person = familyGraph.vertex_collection(name="Person")
else:
    person = familyGraph.create_vertex_collection(name="Person")

# List vertex collections in the graph.
familyGraph.vertex_collections()

['Person']

In [45]:
# Insère les vertices
p1 = person.insert({"_key": "Jane"})
p2 = person.insert({"_key": "Gene", "gender": "female"})
p3 = person.insert({"_key": "Joan", "gender": "female"})
p4 = person.insert({"_key": "Pat",  "gender": "male"})
p5 = person.insert({"_key": "Mike"})

In [46]:
e1 = familyGraph.link("Parent", p1["_id"], p2["_id"])
e2 = familyGraph.link("Parent", p2["_id"], p4["_id"])
e3 = familyGraph.link("Parent", p3["_id"], p4["_id"])
e4 = familyGraph.link("Parent", p5["_id"], p3["_id"])

## Train Network
Lets define a structure for a simple train network.

![trainNetwork](https://github.com/joerg84/Graph_Powered_ML_Workshop/blob/master/img/train_network.png?raw=1)

In [47]:
# Create the graph
if db.has_graph("RailNetwork"):
    railNetworkGraph = db.graph("RailNetwork")
else:
    railNetworkGraph = db.create_graph("RailNetwork")

In [48]:
# create the node
if railNetworkGraph.has_vertex_collection(name="Cities"):
    citiesNode = railNetworkGraph.vertex_collection(name="Cities")
else:
    citiesNode = railNetworkGraph.create_vertex_collection(name="Cities")

# List vertex collections in the graph.
railNetworkGraph.vertex_collections()

['Cities']

In [49]:
# Define de edge
if not railNetworkGraph.has_edge_definition("Connection"):
    connectionEdge = railNetworkGraph.create_edge_definition(
        edge_collection="Connection",
        from_vertex_collections=["Cities"],
        to_vertex_collections=["Cities"]
    )
else:
    connectionEdge = railNetworkGraph.edge_collection("Connection")

# List edge definitions.
railNetworkGraph.edge_definitions()

[{'edge_collection': 'Connection',
  'from_vertex_collections': ['Cities'],
  'to_vertex_collections': ['Cities']}]

In [53]:
# try to creating some documents
h1 = railNetworkGraph.insert_vertex("Cities", {"_key": "Berlin", "country" : "Germany"})
h2 = railNetworkGraph.insert_vertex('Cities', {"_key": "Boston", "country" : "USA"})

In [ ]:
# linking them
print(h1["_id"])
e1 = railNetworkGraph.link('Connection', h1["_id"], h2["_id"], {"distance_km": 6077})

Cities/Berlin


In [58]:
# As we unfortunately cannot travel from Berlin to Boston by train....
berlin = db.collection("Cities").get("Berlin")
railNetworkGraph.delete_edge(e1["_id"])

True

Next let us add more cities and connections:

In [63]:
cities = [
    "Inverness",
    "Aberdeen",
    "Leuchars",
    "StAndrews",
    "Edinburgh",
    "Glasgow",
    "York",
    "Cologne",
    "Carlisle",
    "Birmingham",
    "London",
    "Brussels",
    "Toronto",
    "Winnipeg",
    "Saskatoon",
    "Edmonton",
    "Jasper",
    "Vancouver"
  ]

connections = [
    ( "Inverness", "Aberdeen", 3, 2.5 ),
    ( "Aberdeen", "Leuchars", 1.5, 1 ),
    ( "Leuchars", "Edinburgh", 1.5, 3 ),
    ( "Edinburgh", "Glasgow", 1, 1 ),
    ( "Edinburgh", "York", 3.5, 4 ),
    ( "Glasgow", "Carlisle", 1, 1 ),
    ( "Carlisle", "York", 2.5, 3.5 ),
    ( "Carlisle", "Birmingham", 2.0, 1 ),
    ( "Birmingham", "London", 1.5, 2.5 ),
    ( "Leuchars", "StAndrews", 0.2, 0.2 ),
    ( "York", "London", 1.8, 2.0 ),
    ( "London", "Brussels", 2.5, 3.5 ),
    ( "Brussels", "Cologne", 2, 1.5 ),
    ( "Toronto", "Winnipeg", 36, 35 ),
    ( "Winnipeg", "Saskatoon", 12, 5 ),
    ( "Saskatoon", "Edmonton", 12, 17 ),
    ( "Edmonton", "Jasper", 6, 5 ),
    ( "Jasper", "Vancouver", 12, 13 )
]


for city in cities:
    railNetworkGraph.insert_vertex("Cities", {"_key": city })

for city1, city2, time1, time2  in connections:
    railNetworkGraph.link('Connection', citiesNode.get(city1)["_id"], citiesNode.get(city2)["_id"], {"travel_time": time1})
    railNetworkGraph.link('Connection', citiesNode.get(city2)["_id"], citiesNode.get(city1)["_id"], {"travel_time": time2})


Finally: Our first Graph Traversal

In [66]:
reachabilty_query = """WITH Cities
FOR vertex, edge, path
  IN 1..5 
  OUTBOUND 'Cities/London'
  GRAPH 'RailNetwork'
  FILTER SUM(path.edges[*].travel_time) < 5
  return 
  { 'city': vertex._key,
    'path': CONCAT_SEPARATOR(" -> ", path.edges[*]._to)
  }"""

queryResult = aql.execute(reachabilty_query)
for result in queryResult:
    print("city: " + result["city"])
    print("path: Cities/London -> " + result["path"])
    print()

city: Brussels
path: Cities/London -> Cities/Brussels

city: Cologne
path: Cities/London -> Cities/Brussels -> Cities/Cologne

city: York
path: Cities/London -> Cities/York

city: London
path: Cities/London -> Cities/York -> Cities/London

city: Birmingham
path: Cities/London -> Cities/Birmingham

city: London
path: Cities/London -> Cities/Birmingham -> Cities/London

city: Carlisle
path: Cities/London -> Cities/Birmingham -> Cities/Carlisle

city: Glasgow
path: Cities/London -> Cities/Birmingham -> Cities/Carlisle -> Cities/Glasgow



Next: Shortest Path

In [67]:
shortest_path_query = """FOR p IN OUTBOUND K_SHORTEST_PATHS 'Cities/Aberdeen' TO 'Cities/London'
  GRAPH 'RailNetwork'
      LIMIT 1
      RETURN {
          places: p.vertices[*]._key,
          travelTimes: p.edges[*].travel_time,
          travelTimeTotal: SUM(p.edges[*].travel_time)
      }"""

queryResult = aql.execute(shortest_path_query)
for result in  queryResult:
    print("places: " +  str(result['places']))
    print("intermediate travel times: " +  str(result['travelTimes']))
    print("total travel time: " +  str(result['travelTimeTotal']))
    print()

places: ['Aberdeen', 'Leuchars', 'Edinburgh', 'York', 'London']
intermediate travel times: [1.5, 1.5, 3.5, 1.8]
total travel time: 8.3



In [68]:
# Alternative Shortest path query with more options
shortest_path_query = """FOR p IN OUTBOUND K_SHORTEST_PATHS'Cities/Aberdeen' TO 'Cities/London'
  GRAPH 'RailNetwork'
      OPTIONS {
      weightAttribute: "travel_time",
      defaultWeight: 100
      }
      LIMIT 3
      RETURN {
          places: p.vertices[*]._key,
          travelTimes: p.edges[*].travel_time,
          travelTimeTotal: SUM(p.edges[*].travel_time)
      }"""
queryResult = aql.execute(shortest_path_query)
for result in  queryResult:
    print("places: " +  str(result['places']))
    print("intermediate travel times: " +  str(result['travelTimes']))
    print("total travel time: " +  str(result['travelTimeTotal']))
    print()

places: ['Aberdeen', 'Leuchars', 'Edinburgh', 'York', 'London']
intermediate travel times: [1.5, 1.5, 3.5, 1.8]
total travel time: 8.3

places: ['Aberdeen', 'Leuchars', 'Edinburgh', 'Glasgow', 'Carlisle', 'Birmingham', 'London']
intermediate travel times: [1.5, 1.5, 1, 1, 2, 1.5]
total travel time: 8.5

places: ['Aberdeen', 'Leuchars', 'Edinburgh', 'Glasgow', 'Carlisle', 'York', 'London']
intermediate travel times: [1.5, 1.5, 1, 1, 2.5, 1.8]
total travel time: 9.3

